# 🍕 Memorizza — Juego de memoria visual con pizzas

**Pon a prueba tu memoria!!**

**Instrucciones :**

**Primero...MEMORIZA:** aparece el pedido de un cliente.

**Luego...SELECCIONA:** aparece una cuadrícula de ingredientes y debes pulsar exactamente los que el cliente pidió.

Si aciertas: **subes de nivel** (hasta el nivel 6)hasta **ganar**🥳

Si fallas: **vuelve a empezar**😭

Un juego creado por Ana de las Muelas, Rocío Díaz, Manuela Lipcsey e Irene Macías 

## 1. Instalar y cargar librerías

Instalamos ipywidgets que nos permite crear elementos interactivos (botones, clics)

In [1]:
!pip install ipywidgets
# Instalamos ipywidgets

import random
# Se utiliza para random.sample() genera números y selecciones aleatorias

import time
# Nos sirve para time.sleep(segundos) que pausará la ejecución del programa

from IPython.display import display, HTML, clear_output
# Lo usamos para mostrar cualquier objeto en una celda de HTML, para que lo dibuje no como texto plano
# clear_output borra el contenido que se ha mostrado hasta ahora en una zona concreta de la celda

import ipywidgets as widgets
# Este sirve para hacer elementos interactivos, botones, cajas de texto, casillas

## 2. Lista de Toppings

Creamos la lista global toppings. Cada topping es un diccionario con:
- `id`: número para identificarlos
- `nombre`: el nombre de cada topping que se muestra al jugador.
- `emoji`

In [2]:
toppings = [
    {"id": 1,  "nombre": "Queso","emoji": "\U0001F9C0"},
    {"id": 2,  "nombre": "Pera","emoji": "\U0001F350"},
    {"id": 3,  "nombre": "Mango","emoji": "\U0001F96D"},
    {"id": 4,  "nombre": "Jamon york","emoji": "\U0001F356"},
    {"id": 5,  "nombre": "Bacon","emoji": "\U0001F953"},
    {"id": 6,  "nombre": "Chorizo","emoji": "\U0001F32D"},
    {"id": 7,  "nombre": "Salchichas","emoji": "\U0001F32D"},
    {"id": 8,  "nombre": "Carne picada","emoji": "\U0001F356"},
    {"id": 9,  "nombre": "Sardinas","emoji": "\U0001F41F"},
    {"id": 10, "nombre": "Huevos","emoji": "\U0001F373"},
    {"id": 11, "nombre": "Tomate","emoji": "\U0001F345"},
    {"id": 12, "nombre": "Aceitunas","emoji": "\U0001FAD2"},
    {"id": 13, "nombre": "Pimiento rojo","emoji": "\U0001F336"},
    {"id": 14, "nombre": "Pimiento verde","emoji": "\U0001F336"},
    {"id": 15, "nombre": "Cebolla","emoji": "\U0001F9C5"},
    {"id": 16, "nombre": "Atun","emoji": "\U0001F41F"},
    {"id": 17, "nombre": "Champinones","emoji": "\U0001F344"},
    {"id": 18, "nombre": "Pina","emoji": "\U0001F34D"},
    {"id": 19, "nombre": "Nueces","emoji": "\U0001F330"},
    {"id": 20, "nombre": "Maiz","emoji": "\U0001F33D"},
]

print(f"Total de toppings cargados: {len(toppings)}")
#los codigos de los emojis se han obtenido de la pagina unicode: https://unicode.org/emoji/charts/full-emoji-list.html#1f350

Total de toppings cargados: 20


## 3. Configuración de dificultad por nivel

Existen 6 niveles que van aumentando en dificultad desde el nivel 1 con 2 toppings pedidos por el cliente y 6 opciones en la cuadrícula hasta el nivel 6 con 7 toppings pedidos por el cliente y 16 opciones en la cuadrícula

Definimos un diccionario config_niveles con, para cada nivel, una tupla (toppings_pedidos, opciones):

| Nivel | Ingredientes pedidos | Opciones en el panel |
|---|---|---|
| 1 | 2 | 6 |
| 2 | 3 | 8 |
| 3 | 4 | 10 |
| 4 | 5 | 12 |
| 5 | 6 | 14 |
| 6 | 7 | 16 |

In [3]:
config_niveles = {
    1: (2, 6),
    2: (3, 8),
    3: (4, 10),
    4: (5, 12),
    5: (6, 14),
    6: (7, 16),
}

## 4. Variables globales de estado

Variables guardan el "estado actual de la partida" mientras se juega

In [4]:
nivel_actual = 1
# Nivel del jugador
pedido_actual = []
# Lista vacia que almacenará los toppings pedidos 
seleccion_actual = set()
# Conjunto vacio, el cual no permite duplicados ni tiene orden

## 5. Widgets de interfaz

- `label_nivel`: título con el nivel actual.
- `out_bocadillo`: donde aparece el pedido del cliente (fase de memorización).
- `out_panel`: donde aparece la cuadrícula de ingredientes (fase de selección).
- `out_mensaje`: mensajes de acierto/fallo.

In [5]:
label_nivel   = widgets.HTML(value="<h3>Nivel 1</h3>")
# El widgets.HTML: su función es mostrar el bloque de texto/HTML en pantalla
# Lo del h3 es para el formato de texto
out_bocadillo = widgets.Output()
out_panel     = widgets.Output()
out_mensaje   = widgets.Output()
# El widgets.Output() imagina una caja vacía en la pantalla, para vaciarla y rellenarla de nuevo sin afectar al resto de la página.
# 3 cajas: bocadillo donde aparece y desaparece el pedido del cliente, panel donde aparece la cuadrícula de botones y el mensaje correcto o fallaste.

## 6. Generar el pedido y las opciones de la ronda

1. Usamos random.sample() para escoger, al azar y sin repetir, los ingredientes que el cliente pide.
2. Escogemos ingredientes adicionales (señuelos) del resto de la lista para completar el panel.
3. Juntamos pedido + señuelos y barajamos el orden con random.shuffle(), para que el jugador no pueda guiarse por la posición.

In [6]:
def generar_pedido_y_opciones(nivel):
    # Desempaquetamos la tupla del nivel (está en el paso 3): cuántos pide el cliente, cuántas opciones hay
    n_pedido, n_opciones = config_niveles[nivel]

    # Elegimos al azar los ingredientes que pide el cliente sin repetir
    pedido = random.sample(toppings, n_pedido)
    # Guardamos solo los id en un conjunto para poder filtrar después
    ids_pedido = {t["id"] for t in pedido}

    # Elegimos señuelos: ingredientes que NO están en el pedido
    restantes = [t for t in toppings if t["id"] not in ids_pedido]
    # Nº de señuelos = opciones totales - ingredientes pedidos
    señuelos = random.sample(restantes, n_opciones - n_pedido)

    # Juntamos pedido + señuelos y los mezclamos para que el orden no delate nada
    opciones = pedido + señuelos
    random.shuffle(opciones)

    # Devolvemos las dos cosas que necesita el resto del juego
    return pedido, opciones

## 7. Comprobación de la generación del pedido y opciones
Comprobación de que el pedido aparece sin repeticiones y los números de pedidos y opciones son correctos.

In [7]:
# Comprobamos la generación de pedidos y opciones en los 6 niveles
for nivel in range(1, 7):
    pedido, opciones = generar_pedido_y_opciones(nivel)
    n_pedido, n_opciones = config_niveles[nivel]

    print(f"Nivel {nivel}: {len(pedido)} ingredientes pedidos, "
          f"{len(opciones)} opciones disponibles")

Nivel 1: 2 ingredientes pedidos, 6 opciones disponibles
Nivel 2: 3 ingredientes pedidos, 8 opciones disponibles
Nivel 3: 4 ingredientes pedidos, 10 opciones disponibles
Nivel 4: 5 ingredientes pedidos, 12 opciones disponibles
Nivel 5: 6 ingredientes pedidos, 14 opciones disponibles
Nivel 6: 7 ingredientes pedidos, 16 opciones disponibles


## 8. Fase de memorización: generación del bocadillo

La función mostrar_bocadillo(pedido) dibuja el "bocadillo de texto" del cliente con HTML.

In [8]:
def mostrar_bocadillo(pedido):
    with out_bocadillo:  # Todo lo que se muestre irá dentro de esta caja
        clear_output()   # Borramos lo que hubiera de la ronda anterior
        # Una línea por topping: "🧀 Queso", separadas con <br> (salto de línea HTML)
        texto = "<br>".join(f"{t['emoji']} {t['nombre']}" for t in pedido)
        # Dibujamos el bocadillo: HTML + CSS (borde, esquinas redondeadas, fondo amarillo)
        display(HTML(f'''
            <div style="border:2px solid #444; border-radius:12px; padding:14px;
                        width:260px; background:#fff8e1; font-size:16px;">
                🗨️ <b>El cliente pide:</b><br><br>{texto}
            </div>
        '''))

## 9. Fase de selección (panel interactivo)

La función mostrar_panel_seleccion(opciones) crea un botón seleccionable (ToggleButton) por cada ingrediente del panel. Cada botón guarda el identificador del ingrediente (topping_id) y está enlazado a on_toggle. Así, cuando el usuario selecciona o deselecciona un ingrediente, se añade o se elimina su identificador de seleccion_actual.

In [9]:
def on_toggle(change):
    
    # Se ejecuta automáticamente cada vez que el jugador pulsa o despulsa un ingrediente.
    # Change es qué ha pasado (pulsado/despulsado), owner es a quién le ha pasado y boton guarda el estado del cambio.
    
    boton = change["owner"]                            
    if change["new"]:                                 
        seleccion_actual.add(boton.topping_id)        # True -> el botón ha quedado pulsado, lo añadimos (add) a la selección
    else:                                              
        seleccion_actual.discard(boton.topping_id)    # Flase -> el botón se ha despulsado, lo quitamos (discard) 
    
def mostrar_panel_seleccion(opciones):
    # Lista donde guardamos todos los botones
    botones = []
    # Recorremos los ingredientes para crear un botón para cada uno
    for t in opciones:
        # Creamos un botón por ingrediente, con emoji + nombre
        boton = widgets.ToggleButton(
           description=t["emoji"] + " " + t["nombre"]
    )
        boton.topping_id = t["id"]                     # Etiquetamos el botón con el id de su topping para saber que es lo que se ha seleccionado
        boton.observe(on_toggle, names="value")        # Llamamos on_toggle cuando se selecciona o se deselecciona el botón
        botones.append(boton)                          # Añadimos el botón a la lista

   # Organización de botones
    panel = widgets.GridBox(botones)

   # Creamos el botón de confirmar (punto 10) y al pulsar se comprueba si es correcto
    boton_confirmar = widgets.Button(description="Confirmar pedido ✅", button_style="success")
    boton_confirmar.on_click(comprobar_pedido)

    with out_panel:
        clear_output()                          # Limpiamos el panel anterior
        display(panel, boton_confirmar)         # Mostramos los botones de ingredientes + botón de confirmar

## 10. Comprobar el pedido y gestionar niveles

La función comprobar_pedido() compara el conjunto de ids del pedido con el conjunto de ids seleccionados. Si coinciden sube de nivel (gana en el nivel 6); si no, vuelve al nivel 1.

In [10]:
def comprobar_pedido(boton_confirmar):
    # Necesario porque vamos a cambiar el valor de nivel_actual
    global nivel_actual  # modificar variable global

    # Guardamos el conjunto con los id de lo que pidió el cliente
    ids_pedido = {t["id"] for t in pedido_actual}

    # Borramos el mensaje de la ronda anterior
    with out_mensaje:
        clear_output()

    # Comparamos lo que pidió el cliente con lo que ha seleccionado el jugador
    if seleccion_actual == ids_pedido:

        if nivel_actual == 6:
            # Ha acertado en el último nivel: gana el juego
            with out_mensaje:
                display(HTML("<h3>🎉 ¡Enhorabuena! Completaste los 6 niveles de Memorizza 🍕</h3>"))

            with out_panel:
                clear_output()  # Quitamos el panel de ingredientes

            nivel_actual = 1  # Dejamos el juego listo para otra partida
            return            # Salimos de la función

        else:
            # Ha acertado, pero aún quedan niveles
            with out_mensaje:
                display(HTML(f"<h3 style='color:green;'>✅ ¡Correcto! Subes al nivel {nivel_actual + 1}</h3>"))

            nivel_actual += 1

    else:
        # Ha fallado: vuelve al nivel 1
        with out_mensaje:
            display( HTML("<h3 style='color:red;'>❌ ¡Fallaste el pedido! Vuelves al nivel 1</h3>"))

        nivel_actual = 1

    # Limpiamos el panel de la ronda que acaba de terminar
    with out_panel:
        clear_output()

    # Pequeña pausa para que el jugador pueda leer el mensaje
    time.sleep(2)

    # Empieza la siguiente ronda
    iniciar_ronda()

## 11. Iniciar una ronda y arrancar el juego

iniciar_ronda() encadena todo: genera pedido y opciones, muestra el bocadillo 3 segundos, lo oculta y muestra el panel de selección. El botón **"Iniciar partida"** arranca la primera ronda y el botón **"Terminar partida"** borra todos los elementos del juego y muestra un mensaje de despedida.

In [11]:
def iniciar_ronda():

    # Indicamos que vamos a modificar estas variables creadas fuera de la función,
    # por eso el global
    global pedido_actual, seleccion_actual

    # Empezamos la ronda sin ningún ingrediente seleccionado
    seleccion_actual = set()

    # Generamos el pedido y las opciones del nivel actual
    pedido_actual, opciones = generar_pedido_y_opciones(nivel_actual)

    # Actualizamos el título para que el jugador vea en qué nivel está
    label_nivel.value = f"<h3>Nivel {nivel_actual}</h3>"

    # Limpiamos el panel y el mensaje de la ronda anterior
    with out_panel:
        clear_output()

    with out_mensaje:
        clear_output()

    # FASE DE MEMORIZAR: mostramos el pedido 3 segundos y lo ocultamos
    mostrar_bocadillo(pedido_actual)
    time.sleep(3)

    with out_bocadillo:
        clear_output()

    # FASE DE SELECCIÓN
    mostrar_panel_seleccion(opciones)


def on_inicio(boton):

    # Indicamos que vamos a modificar el nivel_actual
    global nivel_actual

    # Toda partida nueva empieza en el nivel 1
    nivel_actual = 1

    # Iniciamos la primera ronda
    iniciar_ronda()


def on_terminar(boton):

    # Indicamos que vamos a modificar estas variables creadas fuera de la función
    global nivel_actual, pedido_actual, seleccion_actual

    # Dejamos el estado listo por si se quiere empezar otra partida
    nivel_actual = 1
    pedido_actual = []
    seleccion_actual = set()

    # Hacemos desaparecer todos los elementos del juego
    label_nivel.value = ""

    with out_bocadillo:
        clear_output()

    with out_panel:
        clear_output()

    # Mostramos el mensaje de despedida
    with out_mensaje:
        clear_output()
        display(HTML("<h3>👋 ¡Gracias por jugar a Memorizza! 🍕</h3>"))


# Botón que arranca el juego
boton_inicio = widgets.Button(
    description="🍕 Iniciar partida",
    button_style="primary"
)

# Botón que termina la partida
boton_terminar = widgets.Button(
    description="🛑 Terminar partida",
    button_style="danger"
)

# Conectamos los botones con las funciones
boton_inicio.on_click(on_inicio)
boton_terminar.on_click(on_terminar)

# Colocamos los botones uno al lado del otro
botones_partida = widgets.HBox([boton_inicio, boton_terminar])

# Mostramos todos los elementos
display(
    widgets.VBox([
        botones_partida,
        label_nivel,
        out_bocadillo,
        out_panel,
        out_mensaje
    ])
)